In [ ]:
import matplotlib.pyplot as plt

# Data for the pie chart (Grades C and D)
labels = ['C', 'D']
sizes = [19, 9]
colors = ['#F28500', '#FB0303'] # Orange and Red corresponding to each grade

fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(sizes, labels=labels, colors=colors, autopct='%1.0f%%', startangle=90)
ax.axis('equal')

# plt.title('Distribution of Site Performance Grades')

# Save the figure or show it
plt.savefig('pie_chart.png')
# plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from matplotlib.ticker import MaxNLocator  # Imported to handle integer ticks

# Data
grades = ['A (Sain)', 'B (À surveiller)', 'C (À risque)', 'D (Critique)']
previous_month = [0, 0, 22, 5]
current_month = [0, 0, 19, 9]

# Hexes for the colours
grade_colors = ['#2DC937', '#FFEA00', '#F28500', '#FB0303']

# Bar locations and width
x = np.arange(len(grades))  # the label locations
width = 0.35  # the width of the bars

fig, ax = plt.subplots(figsize=(10, 6))
# Using alpha=0.6 (transparency) for the previous month to differentiate the time periods
rects1 = ax.bar(x - width/2, previous_month, width, color=grade_colors, alpha=0.6, edgecolor='black')
rects2 = ax.bar(x + width/2, current_month, width, color=grade_colors, alpha=1.0, edgecolor='black')

# General formatting
ax.set_ylabel('Nombre de postes', fontsize=12)
ax.set_xlabel('Classement des postes', fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(grades, fontsize=11)

# Force the y-axis labels to be whole numbers
ax.yaxis.set_major_locator(MaxNLocator(integer=True))

# Remove the top and right spines (the box)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Custom Legend
prev_patch = mpatches.Patch(facecolor='gray', alpha=0.6, edgecolor='black', label='Mois précédent')
curr_patch = mpatches.Patch(facecolor='gray', alpha=1.0, edgecolor='black', label='Mois actuel')
ax.legend(handles=[prev_patch, curr_patch], fontsize=11)

# Labels to the top of the bars
ax.bar_label(rects1, padding=3, fontsize=10)
ax.bar_label(rects2, padding=3, fontsize=10)

fig.tight_layout()
plt.savefig('comparison_chart.png')
plt.show()

In [ ]:
import plotly.graph_objects as go
import json

# 1. Labels using Unicode bold characters for the active Nodes
# Sources (Mai): C=22, D=5 | Targets (Juin): C=19, D=9 | New Sites (Juin): 1
labels = [
    "𝟐𝟐",  # Node 0: Mai - C (À risque)
    "𝟓",   # Node 1: Mai - D (Critique)
    "𝟏𝟗",  # Node 2: Juin - C (À risque)
    "𝟗",   # Node 3: Juin - D (Critique)
    "𝟏"    # Node 4: New Sites (New contribution to Juin D)
]

# 2. Define the flow (Source to Target) and the Volume (Value)
# To reach Juin C=19 and D=9 from Mai C=22 and D=5, considering 1 new site:
# 19 units stay in C, 3 units move from C to D, 5 units stay in D, and 1 new unit enters D.
sources = [0, 0, 1, 4]
targets = [2, 3, 3, 3]
values =  [19, 3, 5, 1]

# --- 3. DYNAMIC Y-COORDINATE & GEOMETRY CALCULATOR ---
def get_node_geometry(indices, sources, targets, values, gap_ratio=0.15):
    vols = {i: 0 for i in indices}
    for i in indices:
        in_flow = sum(v for t, v in zip(targets, values) if t == i)
        out_flow = sum(v for s, v in zip(sources, values) if s == i)
        vols[i] = max(in_flow, out_flow)

    total_vol = sum(vols.values())
    num_gaps = len(indices) - 1

    gap_size = gap_ratio / num_gaps if num_gaps > 0 else 0
    available_height = 1.0 - gap_ratio

    geometry = {}
    current_y = 0.0

    for i in indices:
        node_height = (vols[i] / total_vol) * available_height if total_vol > 0 else 0
        geometry[i] = {
            'center_y': current_y + (node_height / 2),
            'top_y': current_y,
            'height': node_height
        }
        current_y += node_height + gap_size

    return geometry, total_vol, available_height

geom_left, total_vol, avail_height = get_node_geometry([0, 1, 4], sources, targets, values, gap_ratio=0.20) # Include Node 4 (New Sites) on the left
geom_right, _, _ = get_node_geometry([2, 3], sources, targets, values, gap_ratio=0.20)

all_geoms = {**geom_left, **geom_right}

x_coords = [0.01, 0.01, 0.99, 0.99, 0.01] # Node 4 is a source, so it's on the left
y_coords = [all_geoms[i]['center_y'] for i in range(5)] # Now 5 nodes
# ------------------------------------------

# 4. Node and Link Colours
node_colors = ['#F28500', '#FB0303', '#F28500', '#FB0303', '#000000'] # Changed color for Node 4 (New Sites) to black
link_colors = [
    "rgba(242, 133, 0, 0.4)",  # C -> C
    "rgba(242, 133, 0, 0.4)",  # C -> D
    "rgba(251, 3, 3, 0.4)",     # D -> D
    "rgba(0, 0, 0, 0.4)"       # New Sites -> D (Changed to black)
]

# 5. Create the Sankey diagram
fig = go.Figure(data=[go.Sankey(
    arrangement="fixed",
    node = dict(
      pad = 20,
      thickness = 30,
      line = dict(color = "black", width = 0.5),
      label = labels,
      color = node_colors,
      x = x_coords,
      y = y_coords
    ),
    link = dict(
      source = sources,
      target = targets,
      value = values,
      color = link_colors
    )
)])

# --- 6. ADD STATIC TEXT FOR SPLIT FLOWS ---
for src in set(sources):
    if sources.count(src) > 1:
        current_link_sankey_y = all_geoms[src]['top_y']

        for idx, (s, val) in enumerate(zip(sources, values)):
            if s == src:
                flow_height = (val / total_vol) * avail_height
                flow_center_y_sankey = current_link_sankey_y + (flow_height / 2)
                paper_y = 1.0 - flow_center_y_sankey

                fig.add_annotation(
                    x=0.07,
                    y=paper_y,
                    xref='paper', yref='paper',
                    text=f"<b>{val}</b>",
                    showarrow=False,
                    font=dict(size=10, color="black"),
                    xanchor='left',
                    yanchor='middle'
                )

                current_link_sankey_y += flow_height
# ------------------------------------------

# 7. Formatting and legends
legend_grades = ['<b>A (Sain)</b>', '<b>B (À surveiller)</b>', '<b>C (À risque)</b>', '<b>D (Critique)</b>', '<b>Nouveaux Postes</b>'] # Added legend for new sites
legend_colors = ['#2DC937', '#FFEA00', '#F28500', '#FB0303', '#000000'] # Changed color for new sites to black

for grade, color in zip(legend_grades, legend_colors):
    fig.add_trace(go.Scatter(
        x=[None], y=[None],
        mode='markers',
        marker=dict(size=15, color=color, symbol='square'),
        name=grade
    ))

# Updated Dates
fig.add_annotation(
    x=0.0, y=-0.12,
    xref='paper', yref='paper',
    text='<b>Mai 2026</b>',
    showarrow=False,
    font=dict(size=14, family="DejaVu Sans"),
    xanchor='left'
)

fig.add_annotation(
    x=1.0, y=-0.12,
    xref='paper', yref='paper',
    text='<b>Juin 2026</b>',
    showarrow=False,
    font=dict(size=14, family="DejaVu Sans"),
    xanchor='right'
)

fig.update_layout(
    height=650,
    xaxis=dict(showgrid=False, zeroline=False, visible=False),
    yaxis=dict(showgrid=False, zeroline=False, visible=False),
    plot_bgcolor='white',
    paper_bgcolor='white',

    legend=dict(
        title="&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;<b>Niveau</b>",
        orientation="v",
        yanchor="middle",
        y=0.5,
        xanchor="left",
        x=1.05
    )
)

# 8. Auto-download with HTML
html_content = f"""
<!DOCTYPE html>
<html>
<head>
    <script src=\"https://cdn.plot.ly/plotly-latest.min.js\"></script>
</head>
<body style=\"font-family: sans-serif; padding: 20px;\">
    <h2>Rendering your chart... Check your downloads folder!</h2>
    <div id=\"plotly-div\"></div>
    <script>
        var figure = {fig.to_json()};
        Plotly.newPlot('plotly-div', figure.data, figure.layout).then(function(gd) {{
            Plotly.downloadImage(gd, {{
                format: 'png',
                width: 1000,
                height: 650,
                filename: 'migration_niveaux_mai_juin_2026'
            }});
        }});
    </script>
</body>
</html>
"""

with open("auto_download_sankey.html", "w", encoding="utf-8") as f:
    f.write(html_content)

print("✅ Success! Updated for Mai/Juin 2026.")

fig.show()